# Notebook 17 — Unix CLI for Bioinformatics

**Module:** 01 — Python & Scientific Computing  
**Notebook:** 17 of 20  
**Prerequisites:** Notebook 16  
**Time estimate:** 90 minutes

---
## Step 1 — Motivation

Every bioinformatics pipeline runs on a Unix system — HPC clusters, cloud VMs, and
most bioinformatics software. Track A interviews routinely include Unix questions:
'how do you count sequences in a FASTA file?', 'how do you sort a BED file?',
'what does `awk` do here?'

This notebook covers the commands tested in bioinformatics interviews and the Python
`subprocess` interface for integrating them into pipelines.

---
## Step 2 — Intuition

Unix tools are composable via pipes: the output of one command is the input of the
next. `cat file.fasta | grep '^>' | wc -l` counts FASTA records in three composable
tools that each do one thing.

Python's `subprocess.run` captures this output or replaces the tool with Python code
when the logic is too complex for a one-liner.

---
## Step 3 — Biological Background

**Key bioinformatics file formats and their Unix handling:**

| Format | Extension | Content | Common Unix operations |
|--------|-----------|---------|------------------------|
| FASTA | .fa, .fasta | Sequences | `grep '^>' file.fa | wc -l` |
| FASTQ | .fq, .fastq | Reads + quality | `awk 'NR%4==1' file.fq | wc -l` |
| BED | .bed | Genomic intervals | `sort -k1,1 -k2,2n`, `awk '{print $3-$2}'` |
| VCF | .vcf | Variants | `grep -v '^#'`, `cut -f1-5` |
| SAM/BAM | .sam, .bam | Alignments | SAMtools (not raw Unix) |
| GFF/GTF | .gff, .gtf | Gene features | `awk '$3=="gene"'` |

---
## Step 4 — Mathematical Explanation

Not applicable — this notebook is about Unix tooling.

---
## Step 5 — Computational Explanation

**Critical Unix commands for bioinformatics interviews:**

| Command | What it does | Bioinformatics use |
|---------|-------------|--------------------|
| `grep` | Pattern matching | Find headers, filter VCF comments |
| `awk` | Column manipulation | Extract fields from TSV/BED |
| `sort` | Sort lines | Sort BED by chrom+position |
| `uniq -c` | Count unique lines | Count reads per sample |
| `cut` | Extract columns | Extract specific VCF fields |
| `wc -l` | Count lines | Count records |
| `head/tail` | First/last lines | Preview large files |
| `zcat/gzip` | Compressed files | Most sequencing data is gzipped |

---
## Step 6 — Python Implementation

In [ ]:
import subprocess, tempfile, pathlib, os

# Cell 6.1 — Detect shell availability
result = subprocess.run(["bash", "--version"], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout.split("\n")[0])
else:
    print("bash not found — use Git Bash or WSL on Windows")

In [ ]:
# Cell 6.2 — Create a synthetic FASTA file for practice
fasta_content = """>gene1 Homo sapiens BRCA1 exon 1
ATGCGATCGATCGATCGATCGATCGATCGATCG
ATCGATCGATCGATCGATCGATCG
>gene2 Homo sapiens TP53 exon 5
ATGGAGGAGCCGCAGTCAGATCCTAGCGAGCAG
CAGAAGGAAATCGATCGATCG
>gene3 Mus musculus Brca1 orthologue
ATGCGATCGATCGATCGATCGATCGATCG
>gene4 Homo sapiens KRAS exon 2
ATGACTGAATATAAACTTGTGGTAGTTGGAGCT
"""

tmp_fasta = pathlib.Path(tempfile.mktemp(suffix=".fasta"))
tmp_fasta.write_text(fasta_content)
print(f"Synthetic FASTA: {tmp_fasta}")

In [ ]:
# Cell 6.3 — Common Unix operations via subprocess
def run_shell(cmd: str) -> str:
    """Run a shell command and return stdout as a string."""
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        return f"ERROR: {result.stderr.strip()}"
    return result.stdout.strip()

fasta = str(tmp_fasta)

print("Count FASTA records:")
print(run_shell(f"grep -c '^>' {fasta}"))

print("\nList gene names (first word after >):")
print(run_shell(f"grep '^>' {fasta} | awk '{{print $1}}' | sed 's/>//'"))

print("\nFilter only Homo sapiens sequences:")
print(run_shell(f"grep 'Homo sapiens' {fasta} | wc -l"), "matching headers")

In [ ]:
# Cell 6.4 — argparse: building a CLI tool
cli_content = '''#!/usr/bin/env python3
"""Count sequences in a FASTA file."""
import argparse
import sys
from pathlib import Path


def count_fasta_records(path: Path) -> int:
    """Count records in a FASTA file without loading into memory."""
    count = 0
    with open(path) as f:
        for line in f:
            if line.startswith(">"):
                count += 1
    return count


def main() -> None:
    parser = argparse.ArgumentParser(description="Count sequences in FASTA files.")
    parser.add_argument("files", nargs="+", type=Path, help="FASTA file(s)")
    parser.add_argument("-v", "--verbose", action="store_true", help="Show per-file counts")
    args = parser.parse_args()

    total = 0
    for fasta_file in args.files:
        if not fasta_file.exists():
            print(f"ERROR: {fasta_file} not found", file=sys.stderr)
            sys.exit(1)
        n = count_fasta_records(fasta_file)
        total += n
        if args.verbose:
            print(f"{n:>8}  {fasta_file}")

    print(f"{total:>8}  total")


if __name__ == "__main__":
    main()
'''

import pathlib
repo_root = pathlib.Path.cwd()
while repo_root != repo_root.parent:
    if (repo_root / ".git").exists(): break
    repo_root = repo_root.parent

scripts_dir = repo_root / "scripts"
scripts_dir.mkdir(exist_ok=True)
count_script = scripts_dir / "count_fasta.py"
count_script.write_text(cli_content)
print(f"Written: {count_script}")

# Test it
result = subprocess.run(
    ["python", str(count_script), str(tmp_fasta), "-v"],
    capture_output=True, text=True
)
print(result.stdout)

In [ ]:
# Cell 6.5 — Interview cheat sheet: 10 Unix commands every bioinformatician must know
commands = [
    ("Count FASTA records",     "grep -c '^>' file.fasta"),
    ("Count FASTQ reads",       "awk 'NR%4==1' file.fastq | wc -l"),
    ("Extract header IDs",      "grep '^>' file.fasta | cut -d' ' -f1 | sed 's/>//' "),
    ("Sort BED by chr+pos",     "sort -k1,1 -k2,2n input.bed > sorted.bed"),
    ("Filter VCF comments",     "grep -v '^#' file.vcf | head -20"),
    ("Compute interval lengths","awk '{print $3-$2}' file.bed | sort -n"),
    ("First 5 records (FASTA)", "head -10 file.fasta"),
    ("Unique chromosomes",       "cut -f1 file.bed | sort | uniq"),
    ("Read gzipped FASTQ",      "zcat file.fastq.gz | head -8"),
    ("Count lines in TSV",      "wc -l file.tsv"),
]
print(f"{'Task':<35} Command")
print("-" * 80)
for task, cmd in commands:
    print(f"  {task:<35} {cmd}")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — Render the interview cheat sheet as a formatted table
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(14, 4))
ax.axis("off")
table_data = [[task, cmd] for task, cmd in commands]
tbl = ax.table(
    cellText=table_data,
    colLabels=["Task", "Unix command"],
    cellLoc="left",
    loc="center",
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(8)
tbl.auto_set_column_width([0, 1])
ax.set_title("Track A — Unix bioinformatics cheat sheet", fontsize=11, pad=20)
plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Write a second CLI tool `scripts/fasta_gc.py` that prints gene_id and GC content
   for each record in a FASTA file. Use `argparse` and `compbio_utils.sequence.gc_content`.
2. Without running the code, predict the output of:
   `awk 'NR%4==2 {sum+=length($0); n++} END {print sum/n}' reads.fastq`
3. On a Unix system (or WSL), time `count_fasta.py` against the equivalent
   `grep -c '^>'` on a 100k-record FASTA file. Which is faster and why?
4. Add `count_fasta` as a console script entry point in `pyproject.toml`.

---
## Quiz — Active Recall (Track A Interview Practice)

1. How do you count the number of sequences in a FASTA file? Give two approaches.
2. What does `awk 'NR%4==2'` do when applied to a FASTQ file?
3. What does `sort -k1,1 -k2,2n` do? What is the `n` in `-k2,2n`?
4. You have a VCF file. How do you remove the header lines and count the remaining variants?
5. What is the difference between `grep` and `awk` for column extraction?

---
## Reflection

**Date completed:** ____________________

> *[Can you answer all 5 quiz questions cold? Is fasta_gc.py written and tested? Which Unix command is still unfamiliar?]*

---
*Next: `18_mini_project_compbio_utils_v0.ipynb`*